# Assignment 04 — Miền CIFAR-10 · Notebook 00: Khảo sát và trực quan hóa dữ liệu

**Học phần:** Phát triển các Hệ thống Thông minh — Học viện Công nghệ Bưu chính Viễn thông
**Sinh viên:** Nguyễn Duy Nghĩa · **Mã sinh viên:** B23DCCN600 · **Lớp:** D23CTPM01
**Giảng viên hướng dẫn:** PGS.TS Trần Đình Quế
**Học kỳ:** Học kỳ 1 năm học 2026 – 2027

---

## Mục tiêu của notebook

CIFAR-10 là tập ảnh màu tự nhiên gồm 60 000 ảnh kích thước $32 \times 32 \times 3$ chia đều cho
mười lớp. Trong toàn bộ Assignment 04, đây là miền dữ liệu ảnh khó nhất: nó dùng để kiểm chứng
rằng các kết luận rút ra từ MNIST không phải là hệ quả của một bài toán quá dễ.

Notebook này thực hiện bốn việc, theo đúng thứ tự:

1. Nạp tệp `../data/cifar10.npz` và kiểm tra tính toàn vẹn của dữ liệu (hình dạng, kiểu số,
   miền giá trị).
2. **Kiểm chứng** — chứ không giả định — rằng mười lớp được phân bố cân bằng ở cả nhánh huấn
   luyện lẫn nhánh kiểm thử.
3. Trực quan hóa một lưới $10 \times 10$ mẫu để người đọc thấy tận mắt mức độ đa dạng trong từng
   lớp.
4. Tính hằng số chuẩn hóa **theo từng kênh màu** trên nhánh huấn luyện, là những con số mà hai
   notebook sau sẽ dùng lại nguyên vẹn.

Cuối notebook, báo cáo phân tích định lượng lý do CIFAR-10 khó hơn MNIST rất nhiều, dựa trên các
số đo thống kê thực sự đo được chứ không dựa trên cảm nhận thị giác.

**Hai hình bắt buộc sinh ra ở đây:** `fig_cifar10_class_distribution.png` và
`fig_cifar10_sample_grid.png`.

## 1. Nhập thư viện và cố định hạt giống ngẫu nhiên

Toàn bộ Assignment 04 dùng `RANDOM_SEED = 42` ở mọi nơi có yếu tố ngẫu nhiên, để mọi con số trong
báo cáo tái lập được. Ở notebook này, tính ngẫu nhiên chỉ xuất hiện khi chọn ảnh minh họa và khi
chia tập huấn luyện / kiểm định, nhưng nguyên tắc vẫn được giữ nghiêm ngặt.

In [1]:
import os, json, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_PATH = '../data/cifar10.npz'
FIG_DIR   = '../reports/figures'
REP_DIR   = '../reports'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(REP_DIR, exist_ok=True)

# Tên mười lớp của CIFAR-10 theo đúng thứ tự nhãn 0..9
CLASS_EN = ['airplane', 'automobile', 'bird', 'cat', 'deer',
            'dog', 'frog', 'horse', 'ship', 'truck']
CLASS_VI = ['máy bay', 'ô tô', 'chim', 'mèo', 'hươu',
            'chó', 'ếch', 'ngựa', 'tàu thủy', 'xe tải']

print('NumPy     :', np.__version__)
print('matplotlib:', matplotlib.__version__)
print('Hạt giống :', RANDOM_SEED)
print()
for k, (en, vi) in enumerate(zip(CLASS_EN, CLASS_VI)):
    print(f'  nhãn {k}: {en:<11s} -> {vi}')

NumPy     : 2.4.0
matplotlib: 3.10.8
Hạt giống : 42

  nhãn 0: airplane    -> máy bay
  nhãn 1: automobile  -> ô tô
  nhãn 2: bird        -> chim
  nhãn 3: cat         -> mèo
  nhãn 4: deer        -> hươu
  nhãn 5: dog         -> chó
  nhãn 6: frog        -> ếch
  nhãn 7: horse       -> ngựa
  nhãn 8: ship        -> tàu thủy
  nhãn 9: truck       -> xe tải


## 2. Nạp dữ liệu và kiểm tra tính toàn vẹn

Tệp `cifar10.npz` được dựng lại từ bản gốc của Đại học Toronto bằng script
`src/Assignment 04/download_cifar10.py`. Tệp nén khoảng 150 MB nên không được đưa vào kho mã
nguồn; script tái tạo nó khi cần.

Bốn khóa trong tệp: `x_train` $(50000, 32, 32, 3)$ kiểu `uint8`, `y_train` $(50000,)$,
`x_test` $(10000, 32, 32, 3)$, `y_test` $(10000,)$. Ảnh được lưu theo bố cục **HWC**
(chiều cao, chiều rộng, kênh màu), khác với bố cục **CHW** mà PyTorch yêu cầu; việc hoán vị trục
sẽ được thực hiện ở bước tiền xử lý của hai notebook sau.

Báo cáo kiểm tra tường minh bốn điều kiện: hình dạng đúng, kiểu số là `uint8`, miền giá trị nằm
trọn trong $[0, 255]$, và nhãn nằm trọn trong $\{0, 1, \dots, 9\}$. Nếu một trong bốn điều kiện
sai thì mọi kết quả phía sau đều vô nghĩa, nên việc kiểm tra này không phải là thủ tục hình thức.

In [2]:
assert os.path.exists(DATA_PATH), (
    f'Không tìm thấy {DATA_PATH}. Chạy: python "src/Assignment 04/download_cifar10.py"')

t0 = time.time()
_d = np.load(DATA_PATH)
x_train_raw, y_train_raw = _d['x_train'], _d['y_train'].astype(np.int64)
x_test_raw,  y_test_raw  = _d['x_test'],  _d['y_test'].astype(np.int64)
print(f'Nạp xong trong {time.time()-t0:.1f}s từ {DATA_PATH} '
      f'({os.path.getsize(DATA_PATH)/1e6:.1f} MB trên đĩa)')
print()

print(f'x_train : {str(x_train_raw.shape):<22} dtype={x_train_raw.dtype} '
      f'min={x_train_raw.min()} max={x_train_raw.max()}')
print(f'y_train : {str(y_train_raw.shape):<22} dtype={y_train_raw.dtype} '
      f'min={y_train_raw.min()} max={y_train_raw.max()}')
print(f'x_test  : {str(x_test_raw.shape):<22} dtype={x_test_raw.dtype} '
      f'min={x_test_raw.min()} max={x_test_raw.max()}')
print(f'y_test  : {str(y_test_raw.shape):<22} dtype={y_test_raw.dtype} '
      f'min={y_test_raw.min()} max={y_test_raw.max()}')
print()

checks = {
    'x_train đúng hình dạng (50000, 32, 32, 3)': x_train_raw.shape == (50000, 32, 32, 3),
    'x_test  đúng hình dạng (10000, 32, 32, 3)': x_test_raw.shape == (10000, 32, 32, 3),
    'ảnh lưu kiểu uint8'                       : x_train_raw.dtype == np.uint8,
    'giá trị điểm ảnh nằm trong [0, 255]'       : (x_train_raw.min() >= 0 and x_train_raw.max() <= 255),
    'nhãn nằm trong {0,...,9}'                  : (set(np.unique(y_train_raw)) == set(range(10))
                                                   and set(np.unique(y_test_raw)) == set(range(10))),
    'không có nhãn khuyết'                      : (len(y_train_raw) == len(x_train_raw)
                                                   and len(y_test_raw) == len(x_test_raw)),
}
for name, ok in checks.items():
    print(f"  [{'ĐẠT ' if ok else 'HỎNG'}] {name}")
assert all(checks.values()), 'Dữ liệu không toàn vẹn, dừng lại.'
print()
print(f'Tổng dung lượng trong bộ nhớ: '
      f'{(x_train_raw.nbytes + x_test_raw.nbytes)/1e6:.1f} MB ở dạng uint8, '
      f'{(x_train_raw.nbytes + x_test_raw.nbytes)*4/1e6:.1f} MB nếu đổi sang float32')

Nạp xong trong 1.0s từ ../data/cifar10.npz (169.6 MB trên đĩa)

x_train : (50000, 32, 32, 3)     dtype=uint8 min=0 max=255
y_train : (50000,)               dtype=int64 min=0 max=9
x_test  : (10000, 32, 32, 3)     dtype=uint8 min=0 max=255
y_test  : (10000,)               dtype=int64 min=0 max=9

  [ĐẠT ] x_train đúng hình dạng (50000, 32, 32, 3)
  [ĐẠT ] x_test  đúng hình dạng (10000, 32, 32, 3)
  [ĐẠT ] ảnh lưu kiểu uint8
  [ĐẠT ] giá trị điểm ảnh nằm trong [0, 255]
  [ĐẠT ] nhãn nằm trong {0,...,9}
  [ĐẠT ] không có nhãn khuyết

Tổng dung lượng trong bộ nhớ: 184.3 MB ở dạng uint8, 737.3 MB nếu đổi sang float32


Kết quả in ra xác nhận cả bốn điều kiện đều đạt. Một chi tiết đáng lưu ý về mặt kỹ thuật: 60 000
ảnh ở dạng `uint8` chỉ chiếm khoảng 184 MB, nhưng khi đổi sang `float32` để đưa vào mạng thì con
số đó nhân lên bốn lần, khoảng 737 MB. Đây chính là lý do mô hình NumPy thuần ở notebook 01 phải
huấn luyện trên một tập con: không phải vì thiếu dữ liệu mà vì chi phí tính toán và bộ nhớ trên
CPU.

## 3. Phân phối lớp: kiểm chứng thay vì giả định

Tài liệu giới thiệu CIFAR-10 nói rằng mỗi lớp có đúng 6 000 ảnh, chia thành 5 000 ảnh huấn luyện
và 1 000 ảnh kiểm thử. Báo cáo **không lấy điều đó làm giả định** mà đếm trực tiếp trên dữ liệu
thực tế đang có trên đĩa, vì một tệp `.npz` bị dựng sai thứ tự hoặc bị cắt cụt vẫn nạp được bình
thường mà không báo lỗi gì.

Nếu phân phối đúng là cân bằng thì `accuracy` là một chỉ số công bằng, mọi lớp đóng góp như nhau
vào giá trị trung bình, và `macro F1` sẽ rất gần `accuracy`. Đây là điểm khác biệt quan trọng so
với miền `diabetes` trong cùng assignment, nơi nhãn dương chỉ chiếm khoảng 8,5% nên `accuracy`
cao vẫn có thể che giấu một mô hình kém.

In [3]:
cnt_train = np.bincount(y_train_raw, minlength=10)
cnt_test  = np.bincount(y_test_raw,  minlength=10)

hdr = f"{'Nhãn':>5}{'Tên lớp (EN)':>14}{'Tên lớp (VI)':>14}{'Train':>9}{'Tỉ lệ':>9}{'Test':>8}{'Tỉ lệ':>9}"
print(hdr); print('-' * len(hdr))
for k in range(10):
    print(f'{k:>5}{CLASS_EN[k]:>14}{CLASS_VI[k]:>14}{cnt_train[k]:>9,}'
          f'{100*cnt_train[k]/cnt_train.sum():>8.2f}%{cnt_test[k]:>8,}'
          f'{100*cnt_test[k]/cnt_test.sum():>8.2f}%')
print('-' * len(hdr))
print(f"{'Tổng':>5}{'':>28}{cnt_train.sum():>9,}{100:>8.2f}%{cnt_test.sum():>8,}{100:>8.2f}%")
print()

bal_train = bool(np.all(cnt_train == 5000))
bal_test  = bool(np.all(cnt_test == 1000))
print(f'Mọi lớp train đúng 5000 ảnh : {bal_train}   '
      f'(min={cnt_train.min()}, max={cnt_train.max()}, độ lệch chuẩn={cnt_train.std():.4f})')
print(f'Mọi lớp test  đúng 1000 ảnh : {bal_test}   '
      f'(min={cnt_test.min()}, max={cnt_test.max()}, độ lệch chuẩn={cnt_test.std():.4f})')
print()
imbalance = cnt_train.max() / cnt_train.min()
print(f'Tỉ số mất cân bằng lớn nhất / nhỏ nhất = {imbalance:.4f} '
      f'(giá trị 1,0000 nghĩa là cân bằng tuyệt đối)')
print(f'Độ chính xác của phép đoán ngẫu nhiên đều  = {1/10:.4f} (10,00%)')
print(f'Độ chính xác của phép đoán theo lớp đa số  = '
      f'{cnt_test.max()/cnt_test.sum():.4f} ({100*cnt_test.max()/cnt_test.sum():.2f}%)')

 Nhãn  Tên lớp (EN)  Tên lớp (VI)    Train    Tỉ lệ    Test    Tỉ lệ
--------------------------------------------------------------------
    0      airplane       máy bay    5,000   10.00%   1,000   10.00%
    1    automobile          ô tô    5,000   10.00%   1,000   10.00%
    2          bird          chim    5,000   10.00%   1,000   10.00%
    3           cat           mèo    5,000   10.00%   1,000   10.00%
    4          deer          hươu    5,000   10.00%   1,000   10.00%
    5           dog           chó    5,000   10.00%   1,000   10.00%
    6          frog           ếch    5,000   10.00%   1,000   10.00%
    7         horse          ngựa    5,000   10.00%   1,000   10.00%
    8          ship      tàu thủy    5,000   10.00%   1,000   10.00%
    9         truck        xe tải    5,000   10.00%   1,000   10.00%
--------------------------------------------------------------------
 Tổng                               50,000  100.00%  10,000  100.00%

Mọi lớp train đúng 5000 ảnh : Tru

Phép đếm xác nhận CIFAR-10 cân bằng **tuyệt đối**: cả mười lớp đều có đúng 5 000 ảnh huấn luyện
và 1 000 ảnh kiểm thử, độ lệch chuẩn của vector đếm bằng 0, tỉ số giữa lớp lớn nhất và nhỏ nhất
bằng đúng 1,0000.

Hệ quả trực tiếp cho phần đánh giá ở hai notebook sau: mốc tham chiếu của một bộ phân loại vô
tri là **10,00%**, và vì các lớp cân bằng nên mốc "đoán theo lớp đa số" cũng chỉ là 10,00%, không
có lối tắt thống kê nào để đạt điểm cao. Mọi con số vượt trên 10% đều là tri thức mà mô hình thực
sự học được từ ảnh.

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
xpos = np.arange(10)
labels_vi = [f'{k}\n{CLASS_VI[k]}' for k in range(10)]
palette = sns.color_palette('tab10', 10)

for ax, cnt, tag, total in [(axes[0], cnt_train, 'Nhánh huấn luyện', 50000),
                            (axes[1], cnt_test,  'Nhánh kiểm thử',   10000)]:
    bars = ax.bar(xpos, cnt, color=palette, edgecolor='black', linewidth=0.7)
    mean_cnt = cnt.mean()
    ax.axhline(mean_cnt, color='navy', ls='--', lw=1.4,
               label=f'Trung bình = {mean_cnt:,.0f} ảnh/lớp')
    for bar, c in zip(bars, cnt):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total*0.004,
                f'{c:,}', ha='center', fontsize=9.5)
    ax.set_xticks(xpos); ax.set_xticklabels(labels_vi, fontsize=9)
    ax.set_xlabel('Lớp'); ax.set_ylabel('Số ảnh')
    ax.set_ylim(0, cnt.max() * 1.16)
    ax.set_title(f'{tag}: {cnt.sum():,} ảnh, mỗi lớp {cnt.min():,}–{cnt.max():,}', fontsize=12.5)
    ax.legend(fontsize=9.5, loc='lower right')

fig.suptitle('CIFAR-10: phân phối mười lớp trên nhánh huấn luyện và nhánh kiểm thử\n'
             '(độ lệch chuẩn của vector đếm bằng 0 ở cả hai nhánh: cân bằng tuyệt đối)',
             fontsize=14, y=1.03)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_cifar10_class_distribution.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_cifar10_class_distribution.png',
      f'({os.path.getsize(FIG_DIR + "/fig_cifar10_class_distribution.png")/1024:.1f} KB)')

Đã lưu fig_cifar10_class_distribution.png (88.0 KB)


**Diễn giải hình `fig_cifar10_class_distribution.png`.** Hai bảng cột phẳng tuyệt đối: mười cột
bên trái đều chạm đúng mốc 5 000 và mười cột bên phải đều chạm đúng mốc 1 000, đường ngang trung
bình trùng khít với đỉnh mọi cột. Tỉ lệ giữa hai nhánh là 5:1 ở mọi lớp, nghĩa là nhánh kiểm thử
là một mẫu đại diện trung thực của phân phối huấn luyện chứ không lệch về một nhóm lớp nào.

Sự cân bằng này khiến CIFAR-10 trở thành nền so sánh sạch: khi notebook 02 báo cáo `per_class_accuracy`,
mọi chênh lệch giữa các lớp đều phản ánh **độ khó thị giác** của lớp đó, không phải hệ quả của
việc lớp đó có ít mẫu huấn luyện hơn.

## 4. Lưới mẫu: nhìn tận mắt độ đa dạng trong từng lớp

Mỗi hàng của lưới dưới đây là mười ảnh được lấy ngẫu nhiên từ **cùng một lớp**. Cách trình bày
này quan trọng hơn việc lấy mười ảnh ngẫu nhiên từ toàn tập, vì điều cần thấy không phải là "ảnh
CIFAR-10 trông như thế nào" mà là **mức độ biến thiên bên trong một lớp** — thứ quyết định bài
toán khó hay dễ.

In [5]:
rng_vis = np.random.default_rng(RANDOM_SEED)
fig, axes = plt.subplots(10, 10, figsize=(13.5, 14))

for k in range(10):
    idx_k = np.where(y_train_raw == k)[0]
    pick = rng_vis.choice(idx_k, size=10, replace=False)
    for j in range(10):
        ax = axes[k, j]
        ax.imshow(x_train_raw[pick[j]])
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_edgecolor('#BBBBBB'); s.set_linewidth(0.6)
    axes[k, 0].set_ylabel(f'{k}. {CLASS_VI[k]}', fontsize=11, rotation=0,
                          ha='right', va='center', labelpad=12)

fig.suptitle('CIFAR-10: mười mẫu ngẫu nhiên cho mỗi lớp (ảnh màu 32 x 32 x 3, hiển thị nguyên bản)\n'
             'Mỗi hàng là một lớp; độ biến thiên bên trong hàng chính là độ khó của bài toán',
             fontsize=14, y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.975])
fig.savefig(f'{FIG_DIR}/fig_cifar10_sample_grid.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_cifar10_sample_grid.png',
      f'({os.path.getsize(FIG_DIR + "/fig_cifar10_sample_grid.png")/1024:.1f} KB)')
print()
print('Chỉ số của các ảnh được chọn cho ba lớp đầu (để tái lập):')
rng_check = np.random.default_rng(RANDOM_SEED)
for k in range(3):
    idx_k = np.where(y_train_raw == k)[0]
    print(f'  lớp {k} ({CLASS_VI[k]}): {rng_check.choice(idx_k, size=10, replace=False).tolist()}')

Đã lưu fig_cifar10_sample_grid.png (472.6 KB)

Chỉ số của các ảnh được chọn cho ba lớp đầu (để tái lập):
  lớp 0 (máy bay): [4314, 38753, 4490, 33058, 22062, 21760, 35269, 4653, 10020, 42992]
  lớp 1 (ô tô): [39208, 20353, 18639, 31951, 41254, 9411, 22451, 24725, 27268, 46177]
  lớp 2 (chim): [34630, 37283, 13556, 21922, 17273, 8037, 31293, 44344, 48344, 3226]


**Diễn giải hình `fig_cifar10_sample_grid.png`.** Ba đặc điểm hiện rõ ngay trên lưới và cả ba đều
là nguồn khó khăn cho mô hình:

1. **Nền cảnh không đồng nhất.** Chữ số MNIST luôn là nét trắng trên nền đen tuyệt đối. Ở đây,
   lớp *máy bay* xuất hiện trên nền trời xanh, nền mây trắng, nền đường băng xám; lớp *tàu thủy*
   trên nền biển, nền cảng, nền trời. Nền chiếm phần lớn diện tích 32 x 32 và luôn thay đổi.
2. **Tư thế và tỉ lệ thay đổi mạnh.** Cùng lớp *ngựa*, có ảnh chụp toàn thân nhìn ngang, có ảnh
   chỉ thấy phần đầu chiếm gần trọn khung. Không tồn tại một "khuôn mẫu" chuẩn để so khớp, trong
   khi chữ số viết tay thì gần như luôn nằm giữa khung và có kích thước tương đương nhau.
3. **Che khuất và cắt cụt.** Nhiều ảnh chỉ chứa một phần đối tượng, hoặc đối tượng bị vật khác
   che mất.

Thêm vào đó là các cặp lớp dễ lẫn về mặt ngữ nghĩa và cả về mặt điểm ảnh: *mèo* với *chó*,
*ô tô* với *xe tải*, *hươu* với *ngựa*. Báo cáo dự đoán những cặp này sẽ chiếm phần lớn khối lượng
ngoài đường chéo của ma trận nhầm lẫn ở notebook 02, và sẽ kiểm chứng dự đoán đó bằng số thật.

## 5. Hằng số chuẩn hóa theo từng kênh màu

Hợp đồng tích hợp quy định: chia cho 255 rồi trừ trung bình, chia độ lệch chuẩn, với các hằng số
**học từ nhánh huấn luyện**. Với ảnh xám MNIST, "trung bình" là một số vô hướng. Với ảnh màu, báo
cáo dùng ba cặp $(\mu_c, \sigma_c)$ riêng cho ba kênh đỏ, lục, lam:

$$x^{(c)}_{\text{norm}} = \frac{x^{(c)}/255 - \mu_c}{\sigma_c}, \qquad c \in \{R, G, B\}$$

Lý do dùng hằng số theo kênh thay vì một hằng số chung: ba kênh của ảnh tự nhiên có thống kê khác
nhau một cách hệ thống (bầu trời và mặt nước đẩy kênh lam lên, thảm thực vật đẩy kênh lục lên).
Chuẩn hóa riêng từng kênh đưa cả ba về cùng thang, giúp bước gradient ban đầu không bị một kênh
lấn át.

Hằng số được tính trên **40 000 ảnh của nhánh huấn luyện** sau khi tách 20% làm tập kiểm định
theo `train_test_split(test_size=0.2, stratify=y, random_state=42)` — đúng phép chia mà cả hai
notebook sau sẽ lặp lại. Tính trên toàn bộ 50 000 ảnh sẽ làm rò rỉ thông tin của tập kiểm định
vào bước tiền xử lý.

In [6]:
idx_all = np.arange(len(x_train_raw))
idx_tr, idx_va = train_test_split(idx_all, test_size=0.2,
                                  stratify=y_train_raw, random_state=RANDOM_SEED)
print(f'Chia nhánh huấn luyện: {len(idx_tr):,} ảnh train / {len(idx_va):,} ảnh validation')
print(f'  phân phối lớp train      : {np.bincount(y_train_raw[idx_tr], minlength=10).tolist()}')
print(f'  phân phối lớp validation : {np.bincount(y_train_raw[idx_va], minlength=10).tolist()}')
print()

x_tr_u8 = x_train_raw[idx_tr]                                 # (40000, 32, 32, 3) uint8

# BẮT BUỘC tích lũy ở float64: xem ô bên dưới để thấy vì sao float32 cho kết quả sai
MEAN_C = np.array([x_tr_u8[:, :, :, c].mean(dtype=np.float64) / 255.0 for c in range(3)])
STD_C  = np.array([x_tr_u8[:, :, :, c].std(dtype=np.float64)  / 255.0 for c in range(3)])
MEAN_SCALAR = float(x_tr_u8.mean(dtype=np.float64) / 255.0)
STD_SCALAR  = float(x_tr_u8.std(dtype=np.float64)  / 255.0)

print('Hằng số chuẩn hóa học từ 40 000 ảnh nhánh train (thang [0, 1]):')
print(f"{'Kênh':>8}{'Trung bình mu_c':>20}{'Độ lệch chuẩn sigma_c':>26}")
print('-' * 54)
for c, nm in enumerate(['Đỏ (R)', 'Lục (G)', 'Lam (B)']):
    print(f'{nm:>8}{MEAN_C[c]:>20.10f}{STD_C[c]:>26.10f}')
print('-' * 54)
print(f'{"Gộp ba kênh":>8}{MEAN_SCALAR:>20.10f}{STD_SCALAR:>26.10f}')
print()
print(f'Biên độ chênh lệch giữa kênh sáng nhất và tối nhất: '
      f'{(MEAN_C.max()-MEAN_C.min()):.6f} '
      f'({100*(MEAN_C.max()-MEAN_C.min())/MEAN_SCALAR:.2f}% so với trung bình chung)')

# Đối chiếu với thống kê của tập kiểm thử để chắc chắn hai nhánh cùng phân phối
MEAN_TE = np.array([x_test_raw[:, :, :, c].mean(dtype=np.float64) / 255.0 for c in range(3)])
STD_TE  = np.array([x_test_raw[:, :, :, c].std(dtype=np.float64)  / 255.0 for c in range(3)])
print()
print('Đối chiếu với nhánh kiểm thử (chỉ để kiểm tra, KHÔNG dùng làm hằng số chuẩn hóa):')
for c, nm in enumerate(['R', 'G', 'B']):
    print(f'  kênh {nm}: train mu={MEAN_C[c]:.6f} sigma={STD_C[c]:.6f} | '
          f'test mu={MEAN_TE[c]:.6f} sigma={STD_TE[c]:.6f} | '
          f'chênh lệch mu = {abs(MEAN_C[c]-MEAN_TE[c]):.6f}')

# Kiểm chứng hiệu lực của phép chuẩn hóa trên một mẫu 5000 ảnh (đủ để xác nhận, nhẹ bộ nhớ)
z = (x_tr_u8[:5000].astype(np.float32) / 255.0 - MEAN_C.astype(np.float32)) / STD_C.astype(np.float32)
print()
print(f'Sau chuẩn hóa, trên 5 000 ảnh đầu của nhánh train: '
      f'trung bình = {z.mean(dtype=np.float64):.8f} (kỳ vọng gần 0), '
      f'độ lệch chuẩn = {z.std(dtype=np.float64):.8f} (kỳ vọng gần 1)')
print(f'Miền giá trị sau chuẩn hóa: [{z.min():.4f}, {z.max():.4f}]')
del z

Chia nhánh huấn luyện: 40,000 ảnh train / 10,000 ảnh validation
  phân phối lớp train      : [4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000]
  phân phối lớp validation : [1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]



Hằng số chuẩn hóa học từ 40 000 ảnh nhánh train (thang [0, 1]):
    Kênh     Trung bình mu_c     Độ lệch chuẩn sigma_c
------------------------------------------------------
  Đỏ (R)        0.4910935266              0.2470076651
 Lục (G)        0.4821471334              0.2435435305
 Lam (B)        0.4465753768              0.2616426137
------------------------------------------------------
Gộp ba kênh        0.4732720123              0.2515897319

Biên độ chênh lệch giữa kênh sáng nhất và tối nhất: 0.044518 (9.41% so với trung bình chung)

Đối chiếu với nhánh kiểm thử (chỉ để kiểm tra, KHÔNG dùng làm hằng số chuẩn hóa):
  kênh R: train mu=0.491094 sigma=0.247008 | test mu=0.494214 sigma=0.246653 | chênh lệch mu = 0.003121
  kênh G: train mu=0.482147 sigma=0.243544 | test mu=0.485131 sigma=0.242892 | chênh lệch mu = 0.002984
  kênh B: train mu=0.446575 sigma=0.261643 | test mu=0.450409 sigma=0.261592 | chênh lệch mu = 0.003834



Sau chuẩn hóa, trên 5 000 ảnh đầu của nhánh train: trung bình = -0.00955016 (kỳ vọng gần 0), độ lệch chuẩn = 0.99727629 (kỳ vọng gần 1)
Miền giá trị sau chuẩn hóa: [-1.9882, 2.1263]


### Một cái bẫy số học phải tránh: tích lũy ở `float32`

Cách viết tự nhiên nhất để tính trung bình là đổi toàn bộ ảnh sang `float32` rồi gọi `.mean()`.
Cách đó **cho kết quả sai** trên tập dữ liệu cỡ này, và sai một cách âm thầm.

Nguyên nhân là phép cộng dồn tuần tự trong dấu phẩy động. `float32` chỉ có 24 bit phần định trị,
tức phân giải tương đối khoảng $6 \times 10^{-8}$. Khi tổng tích lũy đã lớn, việc cộng thêm một số
hạng nhỏ không làm tổng thay đổi chút nào vì kết quả làm tròn về đúng giá trị cũ. Với
$40000 \times 32 \times 32 \approx 41$ triệu số hạng cho mỗi kênh, hiện tượng này bão hòa hoàn
toàn và trung bình của cả ba kênh hội tụ về **cùng một giá trị giả**.

Ô mã dưới đây tái hiện lỗi để đối chiếu, rồi xác nhận rằng cách tính ở trên (tích lũy ở
`float64` qua tham số `dtype=np.float64`) cho đúng các giá trị đã công bố của CIFAR-10.

In [7]:
sample = x_train_raw[idx_tr]

# CÁCH SAI: đổi sang float32 rồi lấy trung bình, phép cộng dồn tích lũy ở float32
wrong = (sample.astype(np.float32) / 255.0).mean(axis=(0, 1, 2))
# CÁCH ĐÚNG: ép phép cộng dồn chạy ở float64
right = np.array([sample[:, :, :, c].mean(dtype=np.float64) / 255.0 for c in range(3)])

print(f"{'Kênh':>8}{'Tích lũy float32 (SAI)':>26}{'Tích lũy float64 (ĐÚNG)':>27}{'Sai lệch':>14}")
print('-' * 76)
for c, nm in enumerate(['Đỏ (R)', 'Lục (G)', 'Lam (B)']):
    print(f'{nm:>8}{wrong[c]:>26.8f}{right[c]:>27.8f}{abs(wrong[c]-right[c]):>14.8f}')
print('-' * 76)
print()
print(f'Ba giá trị của cách SAI có bằng nhau không: '
      f'{bool(np.allclose(wrong, wrong[0], atol=1e-7))}  '
      f'(độ lệch chuẩn giữa ba kênh = {wrong.std():.3e})')
print(f'Ba giá trị của cách ĐÚNG có bằng nhau không: '
      f'{bool(np.allclose(right, right[0], atol=1e-7))}  '
      f'(độ lệch chuẩn giữa ba kênh = {right.std():.3e})')
print()
print('Ba kênh của ảnh tự nhiên KHÔNG thể có trung bình trùng nhau tới bảy chữ số thập phân.')
print('Chính dấu hiệu đó tố cáo lỗi: kết quả của cách SAI là hiện vật của phép làm tròn, không')
print('phải là thống kê của dữ liệu.')
print()
print('Đối chiếu với giá trị chuẩn được công bố rộng rãi cho CIFAR-10 '
      '(mean = 0,4914 / 0,4822 / 0,4465):')
ref_pub = np.array([0.4914, 0.4822, 0.4465])
for c, nm in enumerate(['R', 'G', 'B']):
    print(f'  kênh {nm}: tính được {right[c]:.6f} | công bố {ref_pub[c]:.4f} | '
          f'lệch {abs(right[c]-ref_pub[c]):.6f}')
print(f'Sai lệch lớn nhất so với giá trị công bố: {np.abs(right - ref_pub).max():.6f}')
print('(Chênh lệch nhỏ còn lại là do giá trị công bố tính trên cả 50 000 ảnh, còn ở đây chỉ')
print(' tính trên 40 000 ảnh của nhánh train theo đúng quy định chống rò rỉ dữ liệu.)')
del sample

    Kênh    Tích lũy float32 (SAI)    Tích lũy float64 (ĐÚNG)      Sai lệch
----------------------------------------------------------------------------
  Đỏ (R)                0.40959999                 0.49109353    0.08149354
 Lục (G)                0.40959999                 0.48214713    0.07254714
 Lam (B)                0.40959999                 0.44657538    0.03697539
----------------------------------------------------------------------------

Ba giá trị của cách SAI có bằng nhau không: True  (độ lệch chuẩn giữa ba kênh = 0.000e+00)
Ba giá trị của cách ĐÚNG có bằng nhau không: False  (độ lệch chuẩn giữa ba kênh = 1.923e-02)

Ba kênh của ảnh tự nhiên KHÔNG thể có trung bình trùng nhau tới bảy chữ số thập phân.
Chính dấu hiệu đó tố cáo lỗi: kết quả của cách SAI là hiện vật của phép làm tròn, không
phải là thống kê của dữ liệu.

Đối chiếu với giá trị chuẩn được công bố rộng rãi cho CIFAR-10 (mean = 0,4914 / 0,4822 / 0,4465):
  kênh R: tính được 0.491094 | công bố 0.4914 | lệch 

**Diễn giải bảng hằng số.** Ba kênh lệch nhau một cách hệ thống chứ không ngẫu nhiên: kênh đỏ và
kênh lục sáng hơn kênh lam. Đây là dấu vết thống kê của chính nội dung ảnh — bầu trời và mặt
nước xuất hiện rất nhiều trong các lớp *máy bay*, *tàu thủy*, nhưng đối tượng trung tâm ở phần
lớn các lớp còn lại có tông ấm hơn nền.

Điều đáng chú ý thứ hai là độ lệch chuẩn của cả ba kênh đều xấp xỉ 0,25, tức **lớn hơn nhiều** so
với ảnh MNIST. Ở MNIST, phần lớn điểm ảnh bằng 0 (nền đen) nên phân phối điểm ảnh rất lệch và độ
lệch chuẩn nhỏ. Ở CIFAR-10, cường độ điểm ảnh trải gần như đều khắp miền $[0, 1]$: không có điểm
ảnh nào là "nền mặc định". Một mạng tích chập vì thế không thể học mẹo "vùng khác 0 là đối tượng"
mà buộc phải học các đặc trưng thị giác thật sự.

Thống kê nhánh kiểm thử gần như trùng khít với nhánh huấn luyện ở cả ba kênh. Điều này xác nhận
hai nhánh được rút từ cùng một phân phối, nên chênh lệch giữa độ chính xác kiểm định và độ chính
xác kiểm thử ở các notebook sau chỉ có thể do phương sai lấy mẫu hoặc do quá khớp, không do lệch
phân phối.

## 6. Vì sao CIFAR-10 khó hơn MNIST rất nhiều

Phần trên đã cho thấy sự khó khăn bằng mắt. Phần này lượng hóa nó. Báo cáo dùng ba phép đo có thể
tính trực tiếp từ điểm ảnh, không cần huấn luyện mô hình nào:

1. **Độ nhất quán của khuôn mẫu lớp.** Với mỗi lớp, tính ảnh trung bình rồi đo độ lệch chuẩn
   trung bình của các ảnh trong lớp quanh khuôn mẫu đó. Giá trị càng lớn nghĩa là lớp càng không
   có hình dáng chuẩn.
2. **Khả năng phân tách tuyến tính thô.** Đo khoảng cách giữa các ảnh trung bình của các lớp,
   so với độ phân tán bên trong lớp. Tỉ số này là dạng đơn giản của tiêu chuẩn Fisher.
3. **Tỉ lệ điểm ảnh mang tin.** Ở MNIST, hơn 80% điểm ảnh là nền đen thuần túy và có thể bỏ đi
   mà không mất thông tin. Ở CIFAR-10 con số tương ứng gần bằng 0.

In [8]:
# Ảnh trung bình của từng lớp, độ phân tán bên trong lớp và giữa các lớp
means_c, within_c = [], []
for k in range(10):
    xk = x_train_raw[y_train_raw == k].astype(np.float64) / 255.0   # (5000, 32, 32, 3)
    mk = xk.mean(axis=0)
    means_c.append(mk)
    within_c.append(float(np.sqrt(((xk - mk) ** 2).mean())))        # RMS quanh khuôn mẫu lớp
means_c = np.stack(means_c)                                          # (10, 32, 32, 3)
within_c = np.array(within_c)

global_mean = means_c.mean(axis=0)
between = float(np.sqrt(((means_c - global_mean) ** 2).mean()))
fisher_like = between / within_c.mean()

print(f"{'Lớp':>12}{'Độ phân tán trong lớp (RMS)':>32}")
print('-' * 44)
for k in range(10):
    print(f'{CLASS_VI[k]:>12}{within_c[k]:>32.6f}')
print('-' * 44)
print(f'{"Trung bình":>12}{within_c.mean():>32.6f}')
print()
print(f'Độ phân tán GIỮA các khuôn mẫu lớp (RMS) : {between:.6f}')
print(f'Tỉ số giữa-lớp / trong-lớp               : {fisher_like:.4f}')
print()
print('Diễn giải: tỉ số này nhỏ hơn 1 rất nhiều nghĩa là hai ảnh cùng lớp có thể khác nhau')
print('nhiều hơn hai ảnh khác lớp, xét trên không gian điểm ảnh thô. Đây chính là lý do một bộ')
print('phân loại tuyến tính trên điểm ảnh thô hầu như không hoạt động được trên CIFAR-10.')
print()

# Tỉ lệ điểm ảnh gần như không mang tin (nền thuần)
flat_tr = x_train_raw.reshape(len(x_train_raw), -1)
pix_std = flat_tr.std(axis=0)                # độ lệch chuẩn của từng vị trí điểm ảnh
dead = int((pix_std < 1.0).sum())
print(f'Số vị trí điểm ảnh gần như bất biến trên toàn tập (độ lệch chuẩn < 1/255): '
      f'{dead} trên {len(pix_std)} ({100*dead/len(pix_std):.2f}%)')
print(f'Độ lệch chuẩn theo vị trí: nhỏ nhất={pix_std.min():.2f}, '
      f'trung vị={np.median(pix_std):.2f}, lớn nhất={pix_std.max():.2f}')
print()
print('Ở MNIST, các điểm ảnh viền khung luôn bằng 0 nên có hàng trăm vị trí "chết"; ở CIFAR-10')
print('gần như mọi vị trí đều biến thiên, tức mọi điểm ảnh đều có thể mang tin.')

         Lớp     Độ phân tán trong lớp (RMS)
--------------------------------------------
     máy bay                        0.246480
        ô tô                        0.262476
        chim                        0.229619
         mèo                        0.255159
        hươu                        0.210881
         chó                        0.245839
         ếch                        0.221774
        ngựa                        0.243028
    tàu thủy                        0.231945
      xe tải                        0.250784
--------------------------------------------
  Trung bình                        0.239798

Độ phân tán GIỮA các khuôn mẫu lớp (RMS) : 0.065172
Tỉ số giữa-lớp / trong-lớp               : 0.2718

Diễn giải: tỉ số này nhỏ hơn 1 rất nhiều nghĩa là hai ảnh cùng lớp có thể khác nhau
nhiều hơn hai ảnh khác lớp, xét trên không gian điểm ảnh thô. Đây chính là lý do một bộ
phân loại tuyến tính trên điểm ảnh thô hầu như không hoạt động được trên CIFAR-10.



Số vị trí điểm ảnh gần như bất biến trên toàn tập (độ lệch chuẩn < 1/255): 0 trên 3072 (0.00%)
Độ lệch chuẩn theo vị trí: nhỏ nhất=57.42, trung vị=61.53, lớn nhất=80.45

Ở MNIST, các điểm ảnh viền khung luôn bằng 0 nên có hàng trăm vị trí "chết"; ở CIFAR-10
gần như mọi vị trí đều biến thiên, tức mọi điểm ảnh đều có thể mang tin.


**Diễn giải ba phép đo.** Độ phân tán bên trong lớp của CIFAR-10 lớn hơn hẳn độ phân tán giữa các
khuôn mẫu lớp, khiến tỉ số kiểu Fisher nhỏ hơn 1 rõ rệt. Nói cách khác: nếu chỉ so sánh ảnh theo
từng điểm ảnh, hai con mèo khác nhau có thể cách xa nhau hơn là một con mèo với một con chó. Mọi
phương pháp dựa trên khoảng cách trong không gian điểm ảnh thô đều sụp đổ trước đặc điểm này, và
đó chính là lý do phải cần đến biểu diễn phân cấp của mạng tích chập.

Phép đo thứ ba giải thích vì sao ta không thể "cắt bớt" dữ liệu: gần như không có vị trí điểm ảnh
nào bất biến, nên toàn bộ 3 072 chiều đầu vào đều mang tin. Tương phản với MNIST, nơi phần viền
luôn đen và mạng có thể bỏ qua an toàn.

Tổng hợp lại, bốn khác biệt cấu trúc giữa hai miền dữ liệu:

| Khía cạnh | MNIST | CIFAR-10 |
|---|---|---|
| Kích thước đầu vào | $28 \times 28 \times 1 = 784$ | $32 \times 32 \times 3 = 3072$ |
| Nền ảnh | Đen thuần, bất biến | Cảnh tự nhiên, thay đổi liên tục |
| Tư thế đối tượng | Gần như chuẩn hóa, nằm giữa khung | Đổi theo góc chụp, tỉ lệ, hướng |
| Che khuất | Hầu như không có | Thường xuyên, đối tượng hay bị cắt cụt |

Cần nhấn mạnh thêm một yếu tố mà ba phép đo trên không nắm bắt được: **độ phân giải chỉ 32 x 32**.
Ở kích thước này, một con mèo chiếm khoảng 20 x 20 điểm ảnh, không đủ để nhìn rõ hình dáng tai hay
kết cấu lông — những dấu hiệu mà con người dùng để phân biệt mèo với chó. Bài toán khó không chỉ
vì cảnh phức tạp mà còn vì thông tin thực sự bị mất đi khi thu nhỏ ảnh.

Vì vậy, báo cáo **đặt kỳ vọng trước khi chạy thí nghiệm**: mạng CNN NumPy thuần huấn luyện trên
tập con sẽ đạt khoảng 45–55%, còn mạng framework sâu hơn trên toàn bộ dữ liệu đạt khoảng 68–75%.
Cả hai khoảng đều thấp hơn nhiều so với mức trên 98% của MNIST, và điều đó là bình thường chứ
không phải dấu hiệu cài đặt sai. Báo cáo cam kết ghi đúng con số đo được, kể cả khi nó nằm ngoài
khoảng dự kiến.

In [9]:
summary = {
    'domain': 'cifar10',
    'n_train_total': int(len(x_train_raw)),
    'n_test_total': int(len(x_test_raw)),
    'n_classes': 10,
    'class_names_en': CLASS_EN,
    'class_names_vi': CLASS_VI,
    'class_counts_train': cnt_train.tolist(),
    'class_counts_test': cnt_test.tolist(),
    'balanced_train': bal_train,
    'balanced_test': bal_test,
    'split': {'n_train': int(len(idx_tr)), 'n_val': int(len(idx_va)),
              'test_size': 0.2, 'stratify': True, 'random_state': RANDOM_SEED},
    'preprocess': {
        'mean_per_channel': [float(v) for v in MEAN_C],
        'std_per_channel':  [float(v) for v in STD_C],
        'mean_scalar': MEAN_SCALAR, 'std_scalar': STD_SCALAR, 'scale': 255.0,
    },
    'difficulty': {
        'within_class_rms': [float(v) for v in within_c],
        'within_class_rms_mean': float(within_c.mean()),
        'between_class_rms': between,
        'fisher_like_ratio': fisher_like,
        'near_constant_pixels': dead,
        'n_pixel_positions': int(len(pix_std)),
    },
}
out = os.path.join(REP_DIR, 'cifar10_eda_summary.json')
with open(out, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print('Đã ghi', out, f'({os.path.getsize(out)/1024:.1f} KB)')
print()

required = ['fig_cifar10_class_distribution.png', 'fig_cifar10_sample_grid.png']
print('Kiểm tra hai hình bắt buộc của notebook này:')
for f in required:
    p = os.path.join(FIG_DIR, f)
    print(f"  {'OK   ' if os.path.exists(p) else 'THIẾU'} {f:42s} "
          f"{os.path.getsize(p)/1024 if os.path.exists(p) else 0:8.1f} KB")
print()
print('Notebook 00 hoàn tất. Notebook 01 sẽ cài CNN 2D bằng NumPy thuần cho ảnh ba kênh.')

Đã ghi ../reports\cifar10_eda_summary.json (1.8 KB)

Kiểm tra hai hình bắt buộc của notebook này:
  OK    fig_cifar10_class_distribution.png             88.0 KB
  OK    fig_cifar10_sample_grid.png                   472.6 KB

Notebook 00 hoàn tất. Notebook 01 sẽ cài CNN 2D bằng NumPy thuần cho ảnh ba kênh.


## 7. Kết luận của notebook 00

Báo cáo đã xác nhận bằng số liệu thật, không giả định:

- Dữ liệu toàn vẹn: đúng hình dạng, đúng kiểu `uint8`, đúng miền giá trị, đủ mười nhãn.
- Phân phối lớp **cân bằng tuyệt đối** ở cả hai nhánh, nên `accuracy` là chỉ số công bằng và mốc
  ngẫu nhiên là đúng 10,00%.
- Hằng số chuẩn hóa theo từng kênh được tính trên 40 000 ảnh nhánh huấn luyện, có kiểm chứng rằng
  dữ liệu sau chuẩn hóa có trung bình 0 và độ lệch chuẩn 1.
- Ba phép đo định lượng cho thấy độ phân tán bên trong lớp vượt độ phân tán giữa các lớp, và gần
  như không có điểm ảnh nào bất biến — hai đặc điểm khiến CIFAR-10 khó hơn MNIST về bản chất chứ
  không chỉ về cảm nhận.

Hai hình bắt buộc đã được lưu vào `../reports/figures/`. Notebook tiếp theo cài đặt mạng tích chập
hai chiều bằng NumPy thuần cho ảnh ba kênh, kèm kiểm chứng gradient bằng sai phân hữu hạn.